# 🏷️ PHASE 3B: RULE-BASED DETECTION
## Apply SAP Business Rules to Detect Anomalies

**Objective:** Detect anomalies using SAP P2P business logic

**Input:** Features + Labels  
**Output:** Anomaly detection results with risk scores

---

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.insert(0, '../scripts')
from config import RAW_DATA_FILE, PROCESSED_DATA_DIR
from logger import get_logger
from rule_engine import detect_anomalies

logger = get_logger(__name__)

# Visualization
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Imports successful!")

2026-05-25 16:49:44,255 - logger - INFO - 🔧 DEBUG MODE ENABLED
2026-05-25 16:49:44,264 - logger - INFO - 📝 Logs saved to: c:\Users\1boughai\Desktop\IDP-Monitoring-Project\src\notebooks\src\outputs\project_20260525_164944.log


ImportError: cannot import name 'detect_anomalies' from 'rule_engine' (c:\Users\1boughai\Desktop\IDP-Monitoring-Project\src\notebooks\../scripts\rule_engine.py)

## 1️⃣ LOAD DATA & FEATURES

In [ ]:
# Load data with features
features_file = f"{PROCESSED_DATA_DIR}/documents_with_labels_and_features_phase1.csv"

try:
    df = pd.read_csv(features_file)
    print(f"✅ Loaded {len(df):,} records with features")
except FileNotFoundError:
    print(f"⚠️ File not found: {features_file}")
    print("   Please run 04_feature_engineering.ipynb first")

## 2️⃣ ANOMALY DETECTION BY RULES

In [ ]:
# Check for anomaly label column
anomaly_col = None
if 'anomaly_label' in df.columns:
    anomaly_col = 'anomaly_label'
elif 'is_anomaly' in df.columns:
    anomaly_col = 'is_anomaly'

if anomaly_col:
    print(f"\n🏷️ ANOMALY DISTRIBUTION:")
    print(df[anomaly_col].value_counts())
    
    # Calculate percentages
    print(f"\n📊 PERCENTAGES:")
    print(df[anomaly_col].value_counts(normalize=True) * 100)
else:
    print("⚠️ No anomaly column found. Creating basic labels...")
    
    # Create basic rules
    if 'po_history_category_|_bewtp' in df.columns:
        df['has_gr'] = df['po_history_category_|_bewtp'] == 'E'
        df['has_ir'] = df['po_history_category_|_bewtp'] == 'Q'
        
        # Simple rule: OK if both GR and IR present, else ANOMALY
        df['is_anomaly'] = ~(df['has_gr'] & df['has_ir'])
        
        print(f"\nAnomalies detected: {df['is_anomaly'].sum():,}")

## 3️⃣ ANOMALY TYPES

In [ ]:
# Analyze anomaly types
if 'anomaly_severity' in df.columns:
    print("\n⚠️ ANOMALY SEVERITY:")
    print(df['anomaly_severity'].value_counts())
    print(f"\nPercentages:")
    print(df['anomaly_severity'].value_counts(normalize=True) * 100)

if 'anomaly_label' in df.columns:
    print("\n📋 DETAILED ANOMALY LABELS:")
    print(df['anomaly_label'].value_counts())

## 4️⃣ RISK ANALYSIS

In [ ]:
# Calculate financial impact
if 'amount_|_wrbtr' in df.columns and 'is_anomaly' in df.columns:
    anomalies_df = df[df['is_anomaly'] == True]
    total_anomaly_amount = anomalies_df['amount_|_wrbtr'].sum()
    
    print(f"\n💰 FINANCIAL IMPACT:")
    print(f"  Total anomalies: {len(anomalies_df):,}")
    print(f"  Amount at risk: ${total_anomaly_amount:,.2f}")
    print(f"  % of total: {(total_anomaly_amount / df['amount_|_wrbtr'].sum() * 100):.1f}%")

## 5️⃣ TOP ANOMALIES

In [ ]:
# Show top anomalies by supplier
if 'is_anomaly' in df.columns:
    anomalies_df = df[df['is_anomaly'] == True]
    
    # Check for supplier column
    supplier_col = None
    if 'supplier_|_lifnr' in anomalies_df.columns:
        supplier_col = 'supplier_|_lifnr'
    elif 'supplier_name' in anomalies_df.columns:
        supplier_col = 'supplier_name'
    
    if supplier_col:
        print(f"\n🏪 TOP SUPPLIERS WITH ANOMALIES:")
        top_suppliers = anomalies_df[supplier_col].value_counts().head(10)
        print(top_suppliers)
    else:
        print("⚠️ No supplier column found")
    
    # Show largest anomalies
    if 'amount_|_wrbtr' in anomalies_df.columns:
        print(f"\n💸 TOP 10 LARGEST ANOMALIES:")
        top_amounts = anomalies_df.nlargest(10, 'amount_|_wrbtr')[['purchasing_document_|_ebeln', 'amount_|_wrbtr']]
        print(top_amounts)

## 6️⃣ VISUALIZATION

In [ ]:
# Plot anomaly distribution
if 'is_anomaly' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Anomaly counts
    anomaly_counts = df['is_anomaly'].value_counts()
    axes[0].bar(['Normal', 'Anomaly'], [anomaly_counts[False], anomaly_counts[True]])
    axes[0].set_title('Transaction Distribution', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Count')
    
    # Financial impact
    if 'amount_|_wrbtr' in df.columns:
        financial_impact = df.groupby('is_anomaly')['amount_|_wrbtr'].sum()
        axes[1].bar(['Normal', 'Anomaly'], [financial_impact[False], financial_impact[True]])
        axes[1].set_title('Financial Impact', fontsize=14, fontweight='bold')
        axes[1].set_ylabel('Amount ($)')
    
    plt.tight_layout()
    plt.show()
    
    print("✅ Visualization complete")

## ✅ RULE-BASED DETECTION SUMMARY

**Summary:**
- ✅ Applied SAP P2P business rules
- ✅ Detected anomalies by GR/IR matching
- ✅ Calculated financial impact
- ✅ Identified top anomalies
- ✅ Visualized results

**Next Step:** Proceed to 06_model_training.ipynb for ML modeling